In [ ]:
import hashlib
import math


# BLOOM FILTER
def compute_bloom_filter(password: str):
    bloom_filter = [0] * 1000
    padded_pwd = f" {password} "
    bigrams = []

    for i in range(len(padded_pwd) - 1):
        bigrams.append(padded_pwd[i:i + 2])

    for bg in bigrams:
        encoded_bg = bg.encode("utf-8")
        f = int(hashlib.sha256(encoded_bg).hexdigest(), 16) % 1000
        g = int(hashlib.md5(encoded_bg).hexdigest(), 16) % 1000

        for i in range(20):
            h_i = (f + i * g) % 1000
            bloom_filter[h_i] = 1

    return bloom_filter


# JACCARD
def jaccard(b1, b2):
    gamma = 0

    for i in range(len(b1)):
        if b1[i] == 1 and b2[i] == 1:
            gamma += 1

    k1 = sum(b1)
    k2 = sum(b2)
    denom = k1 + k2 - gamma

    if denom != 0:
        return gamma / denom
    else:
        return 0.0


# DICE
def dice(b1, b2):
    gamma = sum(a and b for a, b in zip(b1, b2))
    k1 = sum(b1)
    k2 = sum(b2)

    return (2 * gamma) / (k1 + k2) if (k1 + k2) != 0 else 0.0


# COSINE
def cosine(b1, b2):
    gamma = sum(a and b for a, b in zip(b1, b2))
    k1 = sum(b1)
    k2 = sum(b2)
    denom = math.sqrt(k1) * math.sqrt(k2)

    return gamma / denom if denom != 0 else 0.0


# LOAD DATASET
def load_password_dataset(filename):
    with open(filename, "r", encoding="utf-8", errors="ignore") as file:
        passwords = [line.strip() for line in file if line.strip()]

    return passwords


# PRECOMPUTE BLOOM FILTERS
def precompute_dataset_filters(passwords):
    dataset_filters = []

    for pwd in passwords:
        beta = compute_bloom_filter(pwd)
        dataset_filters.append((pwd, beta))

    return dataset_filters


# MAIN PASSWORD CHECKER
def password_checker(candidate, dataset_filters, threshold):
    candidate_filter = compute_bloom_filter(candidate)

    best_password = None
    best_j_score = 0
    best_d_score = 0
    best_c_score = 0
    highest_score = 0

    for dataset_password, dataset_filter in dataset_filters:
        j_score = jaccard(candidate_filter, dataset_filter)
        d_score = dice(candidate_filter, dataset_filter)
        c_score = cosine(candidate_filter, dataset_filter)

        current_highest = max(j_score, d_score, c_score)

        if current_highest > highest_score:
            highest_score = current_highest
            best_password = dataset_password
            best_j_score = j_score
            best_d_score = d_score
            best_c_score = c_score

        if j_score >= threshold or d_score >= threshold or c_score >= threshold:
            return ("REJECTED", dataset_password, j_score, d_score, c_score)

    return ("ACCEPTED", best_password, best_j_score, best_d_score, best_c_score)


# MAIN PROGRAM
DATASET_FILE = "rockyou_subset_15.txt"
THRESHOLD = 0.6

passwords = load_password_dataset(DATASET_FILE)
dataset_filters = precompute_dataset_filters(passwords)

candidate = input("Enter a candidate password: ")
while len(candidate) < 8 or len(candidate) > 10:
    print("Invalid password length. Password must be between 8 and 10 characters.")
    candidate = input("Please enter another password: ")
    
result, matched_password, j_score, d_score, c_score = password_checker(
    candidate,
    dataset_filters,
    THRESHOLD,
)

# THE JUSTIF TABLE

print("\n" + "="*85)
print("                                 JUSTIF TABLE")
print("="*85)
print(f"{'Original data set':<20} | {'Jaccard coefficient':<18} | {'The Dice':<18} | {'The Cosine'}")
print(f"{'password P':<20} | {'between β(P)&β(UserP)':<18} | {'coefficient':<18} | {'Similarity'}")
print("-" * 85)
print(f"{str(matched_password):<20} | {round(j_score, 4):<18} | {round(d_score, 4):<18} | {round(c_score, 4)}")
print("="*85)


#JUSTIFICATION / PROOF
print("\nJUSTIFICATION (PROOF)")
print(f"Final Decision: {result}")

if result == "REJECTED":
    print(f"Proof: The candidate password (UserP) is structurally too similar to '{matched_password}'.")
    print(f"Based on the JUSTIF table above, the system enforces a strict threshold of {THRESHOLD}.")
    
    reasons = []
    if j_score >= THRESHOLD: reasons.append(f"Jaccard ({round(j_score, 4)})")
    if d_score >= THRESHOLD: reasons.append(f"Dice ({round(d_score, 4)})")
    if c_score >= THRESHOLD: reasons.append(f"Cosine ({round(c_score, 4)})")
    
    print("The password was rejected because the following metrics met or exceeded that threshold:")
    for r in reasons:
        print(f"{r} >= {THRESHOLD}")
else:
    print("Proof: The candidate password (UserP) is structurally distinct from all compromised passwords.")
    print(f"Based on the JUSTIF table above, the closest matching password was '{matched_password}'.")
    print(f"However, all calculated similarity metrics remained strictly below the {THRESHOLD} threshold.")
    print("Therefore, the password is safe to be ACCEPTED.")

